In [ ]:
using MyPackage
using MyPackage.Geometry
using MyPackage.VLM

using Printf

function new_plane()
    plain = Airfoil("../assets/airfoils/Plain/Plain.dat")
    wing = Surface(
        airfoils=[plain, plain],
        b=6.0,
        chord=y -> 2 * (1 - y^2)^0.5,
        sw_center=0.5,
    )
    return Plane([wing])
end

function new_plane2()
    s1223 = Airfoil("../assets/airfoils/S1223/S1223.dat")
    wing = wing = Surface(
        airfoils=[s1223, s1223],
        b=0.850,
        chord=y -> 0.2,
        sw_center=0.5,
    )
    return Plane([wing])
end

function run_example()
    println("\n" * "="^72)
    println("VLM Solver Example")
    println("="^72)

    V_mag = 20.0
    alpha_deg = -5.0
    beta_deg = 0.0
    rho = 1.225
    mu = 1.81 * 10^(-5)
    # S_ref = 0.85*0.2
    S_ref = 3 * pi
    q_inf = 0.5 * rho * V_mag^2

    plane = new_plane()

    println("\nFreestream: V = $(V_mag) m/s, alpha = $(alpha_deg)°, beta = $(beta_deg)°")
    println("Reference area: $(S_ref) m²")
    println("Q inf: $(q_inf)")
    println("CG: $(plane.data.CG)")

    t0 = time()
    FX, FX_dist, FY, FY_dist, FZ, FZ_dist, L, L_dist, D, D_dist, M, M_dist, Ml, Ml_dist, N, N_dist, L_trefftz, L_dist_trefftz, D_trefftz, D_dist_trefftz = VLMSolver(
        plane,
        V_mag,
        (alpha_deg, beta_deg);
        n_chordxspan=[(20, 30)],
        rho=rho,
        epsilon2=10^(-10)
    )
    elapsed = time() - t0

    CL = L[1] / (q_inf * S_ref)
    CD = D[1] / (q_inf * S_ref)
    CL_trefftz = L_trefftz[1] / (q_inf * S_ref)
    CD_trefftz = D_trefftz[1] / (q_inf * S_ref)

    Re = rho * V_mag * plane.surfaces[1].MAC / mu

    println("\nResults")
    println("-"^72)
    @printf("Solve time       : %.3f s\n", elapsed)
    @printf("Reynolds number  : %.1f\n", Re)
    @printf("FX               : %.6f N\n", FX[1])
    @printf("FY               : %.6f N\n", FY[1])
    @printf("FZ               : %.6f N\n", FZ[1])
    @printf("Lift             : %.6f N\n", L[1])
    @printf("Drag             : %.6f N\n", D[1])
    @printf("CL               : %.6f\n", CL)
    @printf("CD               : %.6f\n", CD)
    @printf("Trefftz lift     : %.6f N\n", L_trefftz[1])
    @printf("Trefftz drag     : %.6f N\n", D_trefftz[1])
    @printf("Trefftz CL       : %.6f N\n", CL_trefftz)
    @printf("Trefftz CD       : %.6f N\n", CD_trefftz)

    return FX, FX_dist, FY, FY_dist, FZ, FZ_dist, L, L_dist, D, D_dist, M, M_dist, Ml, Ml_dist, N, N_dist, L_trefftz, L_dist_trefftz, D_trefftz, D_dist_trefftz
end

result = run_example()